# ViMMRC 2.0: MRC sinh câu trả lời với ViT5

Mục tiêu: chuyển nhãn A–D thành văn bản của đáp án đúng, fine-tune ViT5, đánh giá trên dev/test và gọi API `answer_question(question, passage)`. Notebook này chạy được demo offline mà không tải mô hình. Dữ liệu ViMMRC 2.0 có điều kiện truy cập; xem README trước khi huấn luyện.

## 1. Thiết lập

Chạy từ thư mục `mrc-model_Huy` hoặc thư mục gốc của dự án. Để huấn luyện trên GPU/Colab, cài `pip install -r requirements.txt` trước khi bật công tắc ở bước 4.

In [1]:
from pathlib import Path
import json, sys, subprocess

candidates = [Path.cwd(), Path.cwd() / 'mrc-model_Huy',
              Path.cwd() / 'vimmrc-qa-voice' / 'mrc-model_Huy']
WORK_DIR = next((path for path in candidates if (path / 'mrc_data.py').exists()), None)
assert (WORK_DIR / 'mrc_data.py').exists(), 'Run from the repository or mrc-model_Huy folder'
sys.path.insert(0, str(WORK_DIR.resolve()))
from mrc_data import load_examples, evaluate_predictions
from mrc_client import LexicalBaselineClient, MRCClient
print('MRC workspace:', WORK_DIR.resolve())

MRC workspace: D:\Tai Lieu Hoc Tap Cao Hoc\Học máy tiên tiến và ứng dụng\vimmrc-qa-voice\mrc-model_Huy


## 2. Kiểm tra schema

Mỗi bài đọc có `article`, `questions`, `options`, `answers`. Các mảng song song theo câu hỏi; đáp án A–D được đổi thành văn bản lựa chọn tương ứng. Ví dụ nhỏ bên dưới là dữ liệu tự tạo, không thuộc ViMMRC.

In [2]:
sample = load_examples(WORK_DIR / 'examples' / 'sample_vimmrc.json')
assert len(sample) == 2 and sample[0]['answer'] == 'Hà Nội'
for row in sample:
    print({'question': row['question'], 'label': row['answer_letter'], 'answer': row['answer']})

{'question': 'Thủ đô của Việt Nam là gì?', 'label': 'B', 'answer': 'Hà Nội'}
{'question': 'Hà Nội nằm ở miền nào?', 'label': 'C', 'answer': 'miền Bắc'}


In [3]:
DATA_DIR = WORK_DIR.parent / 'data'
paths = {name: DATA_DIR / f'{name}_vimmrc.json' for name in ('train', 'dev', 'test')}
if all(path.exists() for path in paths.values()):
    from train import validate_split_disjoint
    validate_split_disjoint(paths)
    print('Licensed split counts:', {name: len(load_examples(path)) for name, path in paths.items()})
else:
    print('Licensed ViMMRC files unavailable; offline demo remains runnable.')

Licensed split counts: {'train': 3600, 'dev': 564, 'test': 1109}


## 3. Baseline offline để thử giao diện

Baseline chỉ chọn câu trong passage có nhiều từ trùng với câu hỏi. Đây **không phải** mô hình được huấn luyện và không đại diện cho chất lượng ViT5.

In [4]:
baseline = LexicalBaselineClient()
demo_answer = baseline.answer_question('Thành phố nằm ở miền nào?', sample[0]['passage'])
print('Lexical baseline:', demo_answer)
baseline_predictions = [baseline.answer_question(row['question'], row['passage']) for row in sample]
print('Baseline on synthetic sample:', evaluate_predictions(baseline_predictions, sample))

Lexical baseline: Thành phố nằm ở miền Bắc.
Baseline on synthetic sample: {'exact_match': 0.0, 'token_f1': 0.2, 'choice_accuracy': 0.5}


## 4. Fine-tune ViT5 (chỉ khi đã có GPU và quyền truy cập dữ liệu)

`train.py` kiểm tra schema và rò rỉ bài đọc giữa split, đặt seed, dùng dev chọn checkpoint, và chỉ đánh giá test sau huấn luyện. Mặc định batch 1 + tích lũy gradient 16 để giảm bộ nhớ. Các passage dài hơn 512 token bị cắt; đây là giới hạn cần đo và cải thiện trong thí nghiệm tiếp theo. Kết quả sẽ thay đổi theo phần cứng và seed.

In [5]:
RUN_TRAINING = False  # Đổi thành True sau khi cài requirements, có CUDA và dữ liệu hợp lệ
CHECKPOINT = WORK_DIR / 'checkpoints' / 'vit5-vimmrc'
if RUN_TRAINING:
    subprocess.run([sys.executable, str(WORK_DIR / 'train.py'),
                    '--data-dir', str(DATA_DIR), '--output-dir', str(CHECKPOINT)], check=True)
else:
    print('Training disabled. Set RUN_TRAINING=True on GPU/Colab to fine-tune ViT5.')

Training disabled. Set RUN_TRAINING=True on GPU/Colab to fine-tune ViT5.


## 5. Suy luận và đánh giá

Checkpoint do bước 4 tạo mới có thể khởi tạo `MRCClient`. Metric test được ghi vào `test_metrics.json`: Exact Match, token F1 và độ chính xác trắc nghiệm (`choice_accuracy`, ghép câu sinh với lựa chọn có token F1 cao nhất).

In [6]:
if (CHECKPOINT / 'config.json').exists():
    client = MRCClient(CHECKPOINT)
    print('ViT5:', client.answer_question(sample[0]['question'], sample[0]['passage']))
    metrics_file = CHECKPOINT / 'test_metrics.json'
    if metrics_file.exists():
        print('Held-out test metrics:', json.loads(metrics_file.read_text(encoding='utf-8')))
else:
    print('No fine-tuned checkpoint yet; run step 4 on GPU to test MRCClient.')

No fine-tuned checkpoint yet; run step 4 on GPU to test MRCClient.


## 6. Ghi chú thí nghiệm

Ghi lại phiên bản dữ liệu/model, seed, số epoch và ba metric sau khi chạy GPU. ViMMRC là bài toán trắc nghiệm; chuyển đáp án thành văn bản giúp API trả lời tự do, nhưng Exact Match ở đây đo mức khớp với lựa chọn đúng, chưa đo mọi cách diễn đạt hợp nghĩa.